[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/OpenCampus_Demo/blob/main/OC_ImageGen.ipynb)


# 画像生成デモ（Stable Diffusion）

テキスト（日本語／英語）から画像を生成するデモです．  
日本語プロンプトは [Gemini API](https://ai.google.dev/) で英語に翻訳し，[Stable Diffusion v1.5](https://huggingface.co/sd-legacy/stable-diffusion-v1-5) で画像を生成します．

**実行環境**: Google Colab（ランタイム → GPU: T4 推奨）

## セルの進め方
1. **設定**（生成ステップ数・画像サイズなど）
2. **ライブラリのインストール**
3. **ライブラリの読み込み・モデル準備**（`tokens.json` が必要）
4. **Gradio の起動**

> Hugging Face のユーザー認証は **現時点では不要** です（モデルは公開）．  
> 参考モデルページ: https://huggingface.co/sd-legacy/stable-diffusion-v1-5


## 0. 設定

- 生成が遅い／粗い場合は `NUM_INFERENCE_STEPS` を調整してください（多いほど綺麗だが時間がかかる）．
- 変更後は **初期化セル** と **Gradio 起動セル** を再実行してください．


In [ ]:
# Stable Diffusion の生成設定（T4 / 14GB VRAM 向け）
MODEL_ID = "sd-legacy/stable-diffusion-v1-5"
IMAGE_SIZE = 512  # 幅・高さ（px）．大きいほど VRAM を使う
NUM_INFERENCE_STEPS = 25  # 拡散ステップ数（20〜30 程度が無難）
GUIDANCE_SCALE = 7.5  # プロンプトへの追従度
GEMINI_MODEL = "gemini-2.5-flash"  # 日本語→英語翻訳用

print(f"MODEL_ID = {MODEL_ID}")
print(f"IMAGE_SIZE = {IMAGE_SIZE}")
print(f"NUM_INFERENCE_STEPS = {NUM_INFERENCE_STEPS}")
print(f"GUIDANCE_SCALE = {GUIDANCE_SCALE}")
print(f"GEMINI_MODEL = {GEMINI_MODEL}")


## 1. ライブラリのインストール


In [ ]:
# Colab 標準の diffusers / transformers / torch を利用
# google-genai は Gemini 翻訳用（Colab にも入っているが念のため）
!pip install -q -U "diffusers>=0.30.0" "google-genai>=1.0.0"


## 2. ライブラリの読み込み，変数のインスタンス化

`tokens.json`（`gemini` キー）を読み込み，翻訳クライアントと Stable Diffusion を準備します．  
初回はモデルのダウンロードに数分かかります．

Colab で GitHub バッジから開いた場合は，次のいずれかで `tokens.json` を用意してください．
- 左のファイル欄に `tokens.json` をアップロードする
- またはリポジトリを clone したうえで，手元の `tokens.json` を配置する


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import gradio as gr
import torch
from diffusers import StableDiffusionPipeline
from google import genai
from PIL import Image
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 定数・サンプルプロンプト
# ------------------------------------------------------------
TOKENS_PATH = Path("tokens.json")

# Gradio Examples 用（日本語・英語を混ぜる）
SAMPLE_PROMPTS: list[str] = [
    "青空の下で走る柴犬，写真風",
    "夕焼けの未来都市，デジタルイラスト",
    "a watercolor painting of Mount Fuji at dawn",
    "図書館で本を読む猫，やさしいタッチのイラスト",
    "an astronaut riding a horse on Mars, cinematic lighting",
]


def load_tokens(path: Path = TOKENS_PATH) -> dict:
    """API キーを tokens.json から読み込む．

    Args:
        path (Path): トークンファイルのパス

    Returns:
        dict: キー名とトークン文字列の辞書

    Raises:
        FileNotFoundError: ファイルが無い場合
        KeyError: gemini キーが無い場合
    """
    if not path.exists():
        raise FileNotFoundError(
            f"{path.resolve()} が見つかりません．"
            "{\"gemini\": \"...\"} を含む tokens.json を配置してください．"
        )
    with path.open(encoding="utf-8") as f:
        tokens = json.load(f)
    if "gemini" not in tokens or not tokens["gemini"]:
        raise KeyError("tokens.json に gemini キーがありません．")
    return tokens


def resolve_device() -> str:
    """利用可能な推論デバイスを返す．

    Returns:
        str: "cuda" または "cpu"
    """
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU: {name} ({mem_gb:.1f} GB)")
        return "cuda"
    print("GPU が見つかりません．CPU で実行します（非常に時間がかかります）．")
    return "cpu"


def contains_japanese(text: str) -> bool:
    """文字列に日本語（ひらがな・カタカナ・漢字）が含まれるか判定する．

    Args:
        text (str): 判定対象の文字列

    Returns:
        bool: 日本語を含む場合 True
    """
    return bool(re.search(r"[\u3040-\u30ff\u3400-\u9fff]", text))


def translate_prompt_to_english(prompt: str, client: genai.Client, model: str) -> str:
    """日本語プロンプトを Stable Diffusion 向け英語に翻訳する．

    英語のみの場合はそのまま返す．

    Args:
        prompt (str): ユーザー入力プロンプト
        client (genai.Client): Gemini クライアント
        model (str): Gemini モデル名

    Returns:
        str: 英語プロンプト
    """
    if not contains_japanese(prompt):
        return prompt.strip()

    instruction = (
        "Translate the following image-generation prompt into natural English "
        "suitable for Stable Diffusion. "
        "Keep the meaning, style, and details. "
        "Output ONLY the English prompt, with no quotes or explanation.\n\n"
        f"Prompt:\n{prompt}"
    )
    response = client.models.generate_content(model=model, contents=instruction)
    english = (response.text or "").strip().strip('"').strip("'")
    if not english:
        raise RuntimeError("Gemini から翻訳結果を取得できませんでした．")
    return english


def load_pipeline(model_id: str, device: str) -> StableDiffusionPipeline:
    """Stable Diffusion パイプラインを読み込み，デバイスへ配置する．

    Args:
        model_id (str): Hugging Face モデル ID
        device (str): "cuda" または "cpu"

    Returns:
        StableDiffusionPipeline: 推論用パイプライン
    """
    dtype = torch.float16 if device == "cuda" else torch.float32
    print(f"モデルを読み込み中: {model_id} (dtype={dtype})")
    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=dtype,
    )
    # T4 での VRAM 節約
    pipe.enable_attention_slicing()
    pipe = pipe.to(device)
    pipe.set_progress_bar_config(disable=None, leave=False)
    return pipe


def generate_image(
    prompt: str,
    seed: int | None = None,
) -> tuple[Image.Image | None, str]:
    """プロンプトから画像を生成し，使用した英語プロンプトも返す．

    Args:
        prompt (str): 日本語または英語のプロンプト
        seed (int | None): 乱数シード．None または負値ならランダム

    Returns:
        tuple[Image.Image | None, str]: (生成画像 RGB, 実際に使った英語プロンプト／メッセージ)
    """
    prompt = (prompt or "").strip()
    if not prompt:
        return None, "プロンプトを入力してください．"

    english_prompt = translate_prompt_to_english(prompt, gemini_client, GEMINI_MODEL)

    generator = None
    if seed is not None and int(seed) >= 0:
        generator = torch.Generator(device=device).manual_seed(int(seed))

    # ステップ進捗は diffusers 側の tqdm（leave=False 設定済み）
    result = pipe(
        prompt=english_prompt,
        height=IMAGE_SIZE,
        width=IMAGE_SIZE,
        num_inference_steps=int(NUM_INFERENCE_STEPS),
        guidance_scale=float(GUIDANCE_SCALE),
        generator=generator,
    )
    image = result.images[0]

    note = english_prompt
    if contains_japanese(prompt):
        note = f"【翻訳後】{english_prompt}"
    return image, note


def build_demo() -> gr.Blocks:
    """Gradio UI を構築する．

    Returns:
        gr.Blocks: デモ用 UI
    """
    with gr.Blocks(title="画像生成デモ（Stable Diffusion）") as demo:
        gr.Markdown(
            "## テキストから画像を生成\n"
            "日本語または英語で書いてください．日本語は Gemini が英語に翻訳します．"
        )
        with gr.Row():
            with gr.Column(scale=1):
                prompt_in = gr.Textbox(
                    label="プロンプト",
                    lines=3,
                    placeholder="例: 桜並木を歩くロボット，アニメ風",
                )
                seed_in = gr.Number(
                    label="シード（同じ絵を再現したいとき．-1 でランダム）",
                    value=-1,
                    precision=0,
                )
                run_btn = gr.Button("画像を生成", variant="primary")
            with gr.Column(scale=1):
                image_out = gr.Image(label="生成画像", type="pil")
                prompt_out = gr.Textbox(
                    label="実際に使った英語プロンプト",
                    lines=2,
                    interactive=False,
                )

        examples = [[p, -1] for p in SAMPLE_PROMPTS]
        gr.Examples(
            examples=examples,
            inputs=[prompt_in, seed_in],
            label="サンプルプロンプト（クリックして入力欄に入れる）",
        )

        run_btn.click(
            fn=generate_image,
            inputs=[prompt_in, seed_in],
            outputs=[image_out, prompt_out],
        )

    return demo


# ------------------------------------------------------------
# 初期化
# ------------------------------------------------------------
tokens = load_tokens()
gemini_client = genai.Client(api_key=tokens["gemini"])
device = resolve_device()

# ダウンロード進捗の可視化（ファイル単位）
for _ in tqdm(range(1), desc="パイプライン準備", leave=False):
    pipe = load_pipeline(MODEL_ID, device)

print("準備完了．次のセルで Gradio を起動してください．")


## 3. Gradio の実行


In [ ]:
demo = build_demo()
demo.launch(share=True)
